# ESM-2 Fitness Landscape for Pre-emptive AMR Diagnostic Design

**Publication figures notebook**

Three experiments:
1. **Retrospective** — |ESM-2 LLR| vs clinical prevalence (validation)
2. **Prospective** — Emergence order prediction from LLR
3. **Pre-emptive panel design** — Diagnostic panels from LLR alone vs WHO surveillance

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr

# Style
plt.rcParams.update({
    'figure.figsize': (8, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 13,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
sns.set_palette("colorblind")

RESULTS_DIR = Path("../results")
FIGURES_DIR = Path("../results/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Figure 1: |ESM-2 LLR| vs Clinical Prevalence (Retrospective Validation)

Scatter plot with per-organism color coding. Expected: negative correlation
(low |LLR| = conservative substitution = high prevalence).

In [ ]:
# Load retrospective results
df = pd.read_csv(RESULTS_DIR / "retrospective" / "llr_results.csv")
df = df[df["esm2_llr"].notna()].copy()
df["abs_llr"] = df["esm2_llr"].abs()

org_labels = {
    "mtb": "M. tuberculosis",
    "ecoli": "E. coli",
    "saureus": "S. aureus",
    "ngonorrhoeae": "N. gonorrhoeae",
}
df["organism_label"] = df["organism"].map(org_labels)

fig, ax = plt.subplots(figsize=(10, 7))
for org, group in df.groupby("organism_label"):
    ax.scatter(group["abs_llr"], group["prevalence_pct"],
               label=org, s=50, alpha=0.7, edgecolors='white', linewidth=0.5)

# Overall trend line
rho, p = spearmanr(df["abs_llr"], df["prevalence_pct"])
ax.set_xlabel("|ESM-2 LLR| (evolutionary fitness cost)")
ax.set_ylabel("Clinical prevalence (%)")
ax.set_title(f"|ESM-2 LLR| vs Clinical Prevalence\nSpearman ρ = {rho:.3f} (p = {p:.4f})")
ax.legend(title="Organism", fontsize=10)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

plt.savefig(FIGURES_DIR / "fig1_llr_vs_prevalence.png")
plt.savefig(FIGURES_DIR / "fig1_llr_vs_prevalence.pdf")
plt.show()

## Figure 2: Within-Gene Rank Concordance (Prospective Prediction)

Bar chart: for each gene, rank concordance between |LLR| rank and prevalence rank.
0.5 = random, 1.0 = perfect. This is the key novel result.

In [ ]:
# Load prospective analysis
with open(RESULTS_DIR / "prospective" / "prospective_analysis.json") as f:
    prospective = json.load(f)

gene_data = prospective["within_gene"]
genes = list(gene_data.keys())
concordances = [gene_data[g]["rank_concordance"] for g in genes]

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.Set2(np.linspace(0, 1, len(genes)))
bars = ax.bar(range(len(genes)), concordances, color=colors, edgecolor='black', linewidth=0.5)

ax.axhline(0.5, color='red', linestyle='--', alpha=0.7, label='Random baseline')
ax.set_xticks(range(len(genes)))
ax.set_xticklabels([g.replace("_", "\n") for g in genes], rotation=0, fontsize=9)
ax.set_ylabel("Rank Concordance")
ax.set_title("Within-Gene Emergence Order Prediction\n|ESM-2 LLR| rank vs Prevalence rank")
ax.set_ylim(0, 1.05)
ax.legend()

# Add N labels on bars
for bar, g in zip(bars, genes):
    n = gene_data[g]["n"]
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"N={n}", ha='center', va='bottom', fontsize=8)

plt.savefig(FIGURES_DIR / "fig2_rank_concordance.png")
plt.savefig(FIGURES_DIR / "fig2_rank_concordance.pdf")
plt.show()

## Figure 3: Panel Coverage — LLR-ranked vs Prevalence-ranked vs Random

For each gene, compare diagnostic panel coverage at increasing panel sizes.
This is the clinical application figure.

In [ ]:
# Load panel comparison
with open(RESULTS_DIR / "panel_design" / "panel_comparison.json") as f:
    panels = json.load(f)

# Select genes with enough mutations to show meaningful coverage curves
plot_genes = [g for g, d in panels.items() if d["n_known_mutations"] >= 6]

fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharey=True)
axes = axes.flatten()

for idx, gene_key in enumerate(plot_genes[:6]):
    ax = axes[idx]
    data = panels[gene_key]["panel_comparisons"]

    ks = [d["k"] for d in data]
    llr_cov = [d["llr_coverage"] for d in data]
    prev_cov = [d["prevalence_coverage"] for d in data]
    rand_cov = [d["random_coverage_mean"] for d in data]
    rand_std = [d["random_coverage_std"] for d in data]

    ax.plot(ks, prev_cov, 'o-', color='green', label='Prevalence (gold std)', linewidth=2)
    ax.plot(ks, llr_cov, 's-', color='blue', label='ESM-2 |LLR|', linewidth=2)
    ax.fill_between(ks,
                     np.array(rand_cov) - np.array(rand_std),
                     np.array(rand_cov) + np.array(rand_std),
                     alpha=0.2, color='gray')
    ax.plot(ks, rand_cov, '--', color='gray', label='Random', linewidth=1)

    ax.axhline(90, color='red', linestyle=':', alpha=0.5)
    ax.set_title(gene_key.replace("_", " "), fontsize=11)
    ax.set_xlabel("Panel size (k)")
    if idx % 3 == 0:
        ax.set_ylabel("Coverage (%)")

    if idx == 0:
        ax.legend(fontsize=8)

# Hide empty subplots
for idx in range(len(plot_genes), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("Diagnostic Panel Coverage: ESM-2 LLR-ranked vs WHO Prevalence-ranked",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig3_panel_coverage.png")
plt.savefig(FIGURES_DIR / "fig3_panel_coverage.pdf")
plt.show()